# ITSN1 Gene Variants and their Association with Parkinson's Disease
## Information
- Project: ITSN1 Burden
- Date Created: 7/10/25
- Date Last Modified: 3/11/26
- Data: GP2 Release 11 IMP

## Summary

This project explores the *ITSN1* gene variants in the GP2 data.

## Imports

Re-mount resources if not visible in jupyter.

In [ ]:
! wb resource unmount
! wb resource mount

Import the necessary packages 

In [2]:
# System
import os
import sys

# Dataframe and array manipulation
import pandas as pd
import openpyxl
import numpy as np
import math

# Path management
import pathlib
from pathlib import Path
import glob

# Statistics
import statsmodels.api as sm
import scipy
from scipy import stats

# Process management
import shutil
import subprocess

# Workflow management
import papermill as pm

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Date and time
from datetime import datetime

# Get the timestamp
d = datetime.now().strftime("%m%d%Y")
t = datetime.now().strftime("%m%d%Y_%H%M%S")
print("Last iteration date:", d, "EST")
print("Last iteration time:", t, "EST")

# Get package versions
packages = [pd, np, matplotlib, sns, sm, scipy, pm]

print("\nPackage versions:\n")
for p in packages:
    print("-", p.__name__, p.__version__)

Last iteration date: 12112025 EST
Last iteration time: 12112025_032533 EST

Package versions:

- pandas 2.3.2
- numpy 2.2.6
- matplotlib 3.10.1
- seaborn 0.13.2
- statsmodels.api 0.14.4
- scipy 1.15.2
- papermill 2.6.0


### Assign variables for papermill

<div class="alert alert-block alert-info">
<b>Tip:</b> Read papermill documentation here
</div>

[https://papermill.readthedocs.io/en/latest/index.html](https://papermill.readthedocs.io/en/latest/index.html)


<div class="alert alert-block alert-warning">
<b>Example usage:</b> Use the following arguments in an external notebook or in terminal
</div>

```bash
for ANC in EUR AFR EAS AMR AAC AJ CAS MDE SAS FIN; do
  papermill input_notebook.ipynb /path/to/executed/notebooks/output_${ANC}_chr21_GP2_R{RELEASE}_WGS.ipynb \
    -p ANC $ANC \
    -p CHROM chr21 \
    -p GENE ITSN1
done


#### Parameters

<div class="alert alert-block alert-info">
<b>Tip:</b> This notebook uses papermill to externally assign variables.
</div>

<div class="alert alert-block alert-info">
<b>Tip: </b>Below are default parameters to fill variables if not specified outside of the notebook.
</div>

<div class="alert alert-block alert-info">
<b>Tip:</b> Change cell tag to 'parameters' so papermill will recognize them.
</div>

In [4]:
# Parameters
ANC = "EUR"
GENE = "ITSN1"
CHROM = "21"
STARTBP = "33542501"
ENDBP = "33999861"
RELEASE = "11"
DT = "imp"


In [5]:
# Paths

# Home directory
HOME = "/home/jupyter"

# Directory for base results
BASE_RESULTS_DIR = f"{HOME}/{GENE}_R{RELEASE}_Results_{d}"

# Data output directory (for WGS and imputed data)
DT_DIR = f"{BASE_RESULTS_DIR}/{GENE}_{DT.upper()}"

# Work directory for current ANC
WORK_DIR = f"{DT_DIR}/{ANC}"

# Default VEP directory
VEP_DIR = f"{HOME}/vep_data"
os.makedirs(f"{VEP_DIR}/input",exist_ok=True)
os.makedirs(f"{VEP_DIR}/output",exist_ok=True)

# Default workspace directory
DATA_DIR = f"{HOME}/workspace"

# Create directories
dirsToCreate = [BASE_RESULTS_DIR, DT_DIR, WORK_DIR]
for c in dirsToCreate:
    os.makedirs(c, exist_ok=True)
    

## Package Installs

### PLINK

In [6]:
%%bash

mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/plink; then
echo "Plink1.9 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink \n    -------"
wget -N http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 
unzip -o plink_linux_x86_64_20190304.zip
echo -e "\n plink downloaded and unzipped in /home/jupyter/tools \n "

fi

Plink1.9 is already installed in /home/jupyter/tools/


In [7]:
%%bash

mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/plink2; then
echo "Plink2 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink2 \n    -------"
wget -N https://s3.amazonaws.com/plink2-assets/alpha5/plink2_linux_x86_64_20240820.zip
unzip -o plink2_linux_x86_64_20240820.zip
echo -e "\n plink2 downloaded and unzipped in /home/jupyter/tools \n "

fi

Plink2 is already installed in /home/jupyter/tools/


### RVTests

In [9]:
%%bash

#Install RVTESTS: Option 1 (~15min)
if test -e /home/jupyter/tools/rvtests; then

echo "rvtests is already installed"
else
echo "rvtests is not installed"

mkdir /home/jupyter/tools/rvtests
cd /home/jupyter/tools/rvtests

wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz 

tar -zxvf rvtests_linux64.tar.gz
fi

rvtests is already installed


In [10]:
# chmod to make sure you have permission to run the program
! chmod u+x /home/jupyter/tools/plink
! chmod u+x /home/jupyter/tools/plink2
! chmod 777 /home/jupyter/tools/rvtests/executable/rvtest

## Covariate File

In [ ]:
# Define columns to keep
key_cols = ['GP2ID', 'wgs_prune_reason', 'diagnosis','baseline_GP2_phenotype', 'baseline_GP2_phenotype_for_qc', 
            'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset',  "age_at_diagnosis",'race_for_qc',
            'nba', 'wgs', 'wgs_label','nba_label', 'study_type']
# Load master key
key = pd.read_csv(f'{DATA_DIR}/gp2_tier2_eu_release{RELEASE}/clinical_data/master_key_release{RELEASE}_final_vwb.csv', 
                  header=0, sep=',', usecols=key_cols).rename(columns = {
                 'GP2ID': 'IID',
                 'baseline_GP2_phenotype':'phenotype',
                 'biological_sex_for_qc':'SEX', 
                 'age_at_sample_collection':'AGE', 
                 'race_for_qc':'RACE',
                 'age_of_onset':'AAO',
                 'age_at_diagnosis':'AAD'})

# Get total count of samples in release
print('Total samples: ', key.shape[0])
# Show heading of the key
key.head()

In [ ]:
## Subset to keep ANC of interest and only NBA samples
ancestry_key = key[(key['nba_label']==ANC) & (key['nba']==1)].copy()
print(f"Total in {ANC}: ", ancestry_key.shape[0])
ancestry_key.reset_index(drop=True)

In [13]:
# Restrict imputed samples to only those without whole genome sequencing available
print(f"Removing {(ancestry_key['wgs'] == 1).sum()} samples with available whole genome sequencing")

ancestry_key = ancestry_key[ancestry_key['wgs'] != 1]
print(f"Retaining {ancestry_key.shape[0]} nba only samples")

Removing 13184 samples with available whole genome sequencing
Retaining 58218 nba only samples


In [14]:
# Check counts per each 
columns = ['study_type', 'diagnosis','phenotype', 'baseline_GP2_phenotype_for_qc']

for c in columns:
    df_counts = ancestry_key[c].value_counts(dropna=False).to_frame(name='counts')
    df_counts.index.name = c
    print(df_counts, '\n')

                      counts
study_type                  
Case(/Control)         41440
Population Cohort      10636
Prodromal               3915
Brain Bank              1104
Genetically Enriched     788
Monogenic                335 

                                                    counts
diagnosis                                                 
PD                                                   25895
Control                                              13343
Alive without dementia                                8813
prodromal                                             2375
Prodromal                                             1283
...                                                    ...
Dementia with Lewy bodies; Cerebral white matte...       1
Control (history); Microscopic changes of Alzhe...       1
Control (cognitively normal); Microscopic chang...       1
Control (history of mild cognitive impairment);...       1
Parkinson's disease; Microscopic changes of Alz...       1

In [ ]:
keep_studytype = ['Monogenic', 'Case(/Control)', 'Brain Bank', 'Genetically Enriched']
keep_phenotype = ['pd', 'control']

In [ ]:
print("Before filter: ", ancestry_key.shape[0])
ancestry_key = ancestry_key[
    (ancestry_key["study_type"].isin(keep_studytype)) &
    (ancestry_key["phenotype"].str.contains('|'.join(keep_phenotype), case=False))
]
print("After filter: ", ancestry_key.shape[0])

Before filter:  58218


After filter:  41872


In [17]:
for c in columns:
    df_counts = ancestry_key[c].value_counts(dropna=False).to_frame(name='counts')
    df_counts.index.name = c
    print(df_counts, '\n')

                      counts
study_type                  
Case(/Control)         40326
Genetically Enriched     721
Brain Bank               537
Monogenic                288 

                                                    counts
diagnosis                                                 
PD                                                   25895
Control                                              13289
Parkinson's disease                                    567
Parkinson's Disease                                    444
Parkinson's Disease - Clinically Definite (Bowe...     302
...                                                    ...
Control; Microscopic changes of Alzheimer's dis...       1
Control; Neurofibrillary tangles, mesial tempor...       1
Control; Microscopic changes of Alzheimer's dis...       1
Control; Acute, lacunar-sized hemorrhagic infar...       1
Control (History); Microscopic changes of Alzhe...       1

[287 rows x 1 columns] 

             counts
phenotype  

In [18]:
# Convert phenotype to binary (1/2) and drop NAs

pheno_mapping = {"PD": 2, "Affected_PD":2, "Control": 1}
ancestry_key['PHENO'] = ancestry_key['phenotype'].map(pheno_mapping).astype('Int64')

# drop NAs
ancestry_key = ancestry_key[ancestry_key['PHENO'].notna()]

# Check value counts of pheno
ancestry_key['PHENO'].value_counts(dropna=False)

PHENO
2    28059
1    13792
Name: count, dtype: Int64

In [19]:
# Load information about related individuals 
try:
    print("Before: ", ancestry_key.shape[0])
    related_df = pd.read_csv(f'{DATA_DIR}/gp2_tier2_eu_release{RELEASE}/meta_data/related_samples/{ANC}_release{RELEASE}_vwb.related')
    print("Related detected: ", related_df.shape[0])
    # Make a list of just one set of related people
    related_list = list(related_df['IID1'])

    # remove related individuals
    print(f"Removing {len(related_list)} individuals.")

    ancestry_key = ancestry_key[~ancestry_key["IID"].isin(related_list)]

    # Check size
    print("Unrelated dataset: ", ancestry_key.shape[0])
except:
    print("Warning. No related samples file found. Removing 0 samples.")


Before:  41851


Related detected:  2857
Removing 2857 individuals.
Unrelated dataset:  41021


In [20]:
ancestry_key.shape[0]

41021

In [21]:
# Assign conditions so female=2 and men=1, and -9 otherwise (matching PLINK convention)
# Female = 2; Male = 1
sex_mapping = {"Female": 2, "Male": 1}
ancestry_key["SEX"] = ancestry_key['SEX'].map(sex_mapping).astype('Int64')

# Check value counts of SEX
ancestry_key["SEX"].value_counts(dropna=False)

SEX
1    24008
2    17013
Name: count, dtype: Int64

In [ ]:
## Get the PCs
pcs = pd.read_csv(f'{DATA_DIR}/gp2_tier2_eu_release{RELEASE}/raw_genotypes/{ANC}/{ANC}_release{RELEASE}_vwb.eigenvec', sep='\t')
selected_columns = ['IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
pcs = pd.DataFrame(data=pcs.iloc[:, 1:7].values, columns=selected_columns)

# Drop the first row (since it's now the column names)
pcs = pcs.drop(0)

# Reset the index to remove any potential issues
pcs = pcs.reset_index(drop=True)

# Display the resulting DataFrame
print(pcs)

In [23]:
## Check for samples missing pcs and pheno
df = pd.merge(pcs, ancestry_key, on='IID', how="inner")

print(f"Dropping {ancestry_key.shape[0] - df.shape[0]} samples with no pcs")


## Drop lines with missing pheno
print(f"Dropping {(df['PHENO'].isna()).sum()} samples with missing pheno")

df = df[df['PHENO'].notna()]

df["PHENO"].value_counts(dropna=False)


Dropping 0 samples with no pcs
Dropping 0 samples with missing pheno


PHENO
2    27691
1    13330
Name: count, dtype: Int64

In [ ]:
# get FID from .psam file 
if RELEASE == str(10):
    FID_df = pd.read_csv(f"{DATA_DIR}/gp2_tier2_eu_release{RELEASE}/imputed_genotypes/{ANC}/chr{CHROM}_{ANC}_release{RELEASE}_vwb.psam", sep = "\t", usecols=["IID"], 
                         names=['IID', 'SEX', 'PHENO1'])
    FID_df.insert(0, "#FID", 0)
    df = df.merge(FID_df, how="left", on="IID")
elif RELEASE == str(11):
    FID_df = pd.read_csv(f"{DATA_DIR}/gp2_tier2_eu_release{RELEASE}/imputed_genotypes/{ANC}/chr{CHROM}_{ANC}_release{RELEASE}_vwb.psam", sep = "\t", usecols=["#FID", "IID"])
    df = df.merge(FID_df, how="left", on="IID")
df.head()

In [25]:
## Clean up and keep columns we need 
final_df = df[['#FID','IID', 'SEX', 'AGE', 'PHENO', 'AAO', 'AAD', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']].copy()
final_df.groupby(['PHENO'])['SEX'].value_counts(dropna=False)

PHENO  SEX
1      2       6730
       1       6600
2      1      17408
       2      10283
Name: count, dtype: Int64

In [ ]:
## Make file of sample IDs to keep 
samples_toKeep = final_df[['#FID', 'IID']].copy()

samples_toKeep.to_csv(f'{WORK_DIR}/{ANC}.samplestoKeep', sep = '\t', index=False, header=None)


In [27]:
## Save your covariate file
final_df.to_csv(f'{WORK_DIR}/{ANC}_covariate_file.txt', sep = '\t', index=False, na_rep= "NA", header=True)

### Get cohort summary stats

In [29]:
print(f"""
Number of {ANC} PD cases: {(final_df["PHENO"] == 2).sum()}
Number of {ANC} Controls: {(final_df["PHENO"] == 1).sum()}

PD age range: {round(final_df[final_df["PHENO"] == 2]["AGE"].mean(),2)} +/- {round(final_df[final_df["PHENO"] == 2]["AGE"].std(),2)}
Controls age range: {round(final_df[final_df["PHENO"] == 1]["AGE"].mean(),2)} +/- {round(final_df[final_df["PHENO"] == 1]["AGE"].std(),2)}

Cases in AAO analysis: {((final_df["PHENO"] == 2) & (final_df["AAO"].notna())).sum()}

nan_counts:\n{final_df.isna().sum()}
""")


Number of EUR PD cases: 27691
Number of EUR Controls: 13330

PD age range: 68.45 +/- 9.9
Controls age range: 62.47 +/- 13.43

Cases in AAO analysis: 22147

nan_counts:
#FID         0
IID          0
SEX          0
AGE       7747
PHENO        0
AAO      18874
AAD      29348
PC1          0
PC2          0
PC3          0
PC4          0
PC5          0
dtype: int64



## Subset PLINK ANC file

In [30]:
PLINK_PREFIX = f"chr{CHROM}_{GENE}"
print(PLINK_PREFIX)

chr21_ITSN1


In [31]:
## extract region of interest from file, flanked by 100kb both sides
! /home/jupyter/tools/plink2 \
--pfile {DATA_DIR}/gp2_tier2_eu_release{RELEASE}/imputed_genotypes/{ANC}/chr{CHROM}_{ANC}_release{RELEASE}_vwb \
--chr {CHROM} \
--from-bp {STARTBP} \
--to-bp {ENDBP} \
--make-pgen erase-dosage \
--out {WORK_DIR}/{PLINK_PREFIX}


PLINK v2.0.0-a.6.24LM 64-bit Intel (15 Sep 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_ITSN1.log.
Options in effect:
  --chr 21
  --from-bp 33542501
  --make-pgen erase-dosage
  --out /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_ITSN1
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/EUR/chr21_EUR_release11_vwb
  --to-bp 33999861

Start time: Thu Dec 11 03:25:42 2025
12962 MiB RAM detected, ~9505 available; reserving 6481 MiB for main workspace.
Using up to 2 compute threads.


71402 samples (31594 females, 39808 males; 71402 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/EUR/chr21_EUR_release11_vwb.psam.


1621813 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/EUR/chr21_EUR_release11_vwb.pvar.


1 binary phenotype loaded (32952 cases, 15388 controls).
21785 variants remaining after main filters.
Writing /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_ITSN1.psam
... done.
Writing /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_ITSN1.pvar
... 0%

0%1%1%2%2%3%3%4%4%5%5%6%6%7%7%8%8%9%9%10%10%11%11%12%12%13%13%14%14%15%15%16%16%17%17%18%18%19%20%20%21%21%22%22%23%23%24%24%25%25%26%26%27%27%28%28%29%29%30%30%31%31%32%32%33%33%34%34%35%35%36%36%37%37%38%38%39%40%40%41%41%42%42%43%43%44%44%45%45%46%46%47%47%48%48%49%49%50%50%51%51%52%52%53%53%54%54%55%55%56%56%57%57%58%58%59%60%60%61%61%62%62%63%63%64%64%65%65%66%66%67%67%68%68%69%69%70%70%71%71%72%72%73%73%74%74%75%75%76%76%77%77%78%78%79%80%80%81%81%82%82%83%83%84%84%85%85%86%86%87%87%88%88%

done.
End time: Thu Dec 11 03:25:48 2025


In [32]:
# filter variants to just including those with mac1, and by samples to Keep to make main file for analyses
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/{PLINK_PREFIX} \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--mac 1 \
--make-pgen \
--out {WORK_DIR}/chr{CHROM}_{ANC}

PLINK v2.0.0-a.6.24LM 64-bit Intel (15 Sep 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR.log.
Options in effect:
  --keep /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR.samplestoKeep
  --mac 1
  --make-pgen
  --out /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR
  --pfile /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_ITSN1

Start time: Thu Dec 11 03:25:49 2025
12962 MiB RAM detected, ~9493 available; reserving 6481 MiB for main workspace.
Using up to 2 compute threads.
71402 samples (31594 females, 39808 males; 71402 founders) loaded from
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_ITSN1.psam.
21785 variants loaded from
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_ITSN1.pvar.
1 binary phenotype loaded (32952 cases, 15388 controls).
--keep: 41021 samples remaining.
410

done.
4408 variants removed due to allele frequency threshold(s)
(--maf/--max-maf/--mac/--max-mac).
17377 variants remaining after main filters.
Writing /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR.psam
... done.
Writing /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR.pvar
... 0%0%1%1%2%2%3%3%4%4%5%5%6%6%7%7%8%8%9%9%10%10%11%11%12%12%13%13%14%14%15%15%16%16%17%17%18%18%19%19%20%20%21%21%22%22%23%23%24%24%25%25%26%26%27%27%28%28%29%29%30%30%31%31%32%32%33%33%34%34%35%35%36%36%37%37%38%38%39%39%40%40%41%41%42%42%43%43%44%44%45%45%46%46%47%47%48%48%49%49%50%50%51%51%52%52%53%53%54%54%55%55%56%56%57%57%58%58%59%59%60%

done.
End time: Thu Dec 11 03:25:52 2025


In [33]:
## generate individual pfile for variant annotation

# Get one sample 
samples = pd.read_csv(f"{WORK_DIR}/{ANC}.samplestoKeep", sep = "\t", header=None)
sample_id_1 = samples.iloc[0,0]
sample_id_2 = samples.iloc[0,1]
sample_id = f"{sample_id_1} {sample_id_2}"


# convert to vcf for annotation
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/chr{CHROM}_{ANC} \
--recode vcf id-paste=iid \
--indv {sample_id} \
--out {WORK_DIR}/{ANC}_vep_input

PLINK v2.0.0-a.6.24LM 64-bit Intel (15 Sep 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_vep_input.log.
Options in effect:
  --export vcf id-paste=iid
  --indv AAPDGC_000147 AAPDGC_000147
  --out /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_vep_input
  --pfile /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR

Start time: Thu Dec 11 03:25:53 2025
12962 MiB RAM detected, ~9485 available; reserving 6481 MiB for main workspace.
Using up to 2 compute threads.
41021 samples (17013 females, 24008 males; 41021 founders) loaded from
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR.psam.
17377 variants loaded from
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR.pvar.
1 binary phenotype loaded (26713 cases, 13330 controls).
--indv: 1 sample remaining.
1 sample (0 females, 1 male; 1 founder) remaini

47%48%48%49%49%50%50%51%51%52%52%53%53%54%54%55%55%56%56%57%57%58%58%59%59%60%60%61%61%62%62%63%63%64%64%65%65%66%66%67%67%68%68%69%69%70%70%71%71%72%72%73%73%74%74%75%75%76%76%77%77%78%78%79%79%80%80%81%81%82%82%83%83%84%84%85%85%86%86%87%87%88%88%89%89%90%90%91%91%92%92%93%93%94%94%95%95%96%96%97%97%98%98%99%done.
End time: Thu Dec 11 03:25:53 2025


In [34]:
# move to input directory
os.makedirs(f"{VEP_DIR}/input/R{RELEASE}_{DT}", exist_ok=True)
! cp {WORK_DIR}/{ANC}_vep_input.vcf {VEP_DIR}/input/R{RELEASE}_{DT}/{ANC}_vep_input.vcf

In [35]:
### Bgzip and Tabix (zip and index the file)
! bgzip -k -f {WORK_DIR}/{ANC}_vep_input.vcf
! tabix -f -p vcf {WORK_DIR}/{ANC}_vep_input.vcf.gz
! tabix -f -p vcf {WORK_DIR}/{ANC}_vep_input.vcf.gz

## Annotation using VEP
- *ITSN1* from NCBI gene, with gene region flanked by 200kb on both sides
- hg38 (chr21:33442501-34099861) 

In [36]:
# run container to generate VEP output
vep_cmd = [
    "vep",
    "--cache",
    "--offline",
    "--force_overwrite",
    "--input_file", f"{VEP_DIR}/input/R{RELEASE}_{DT}/{ANC}_vep_input.vcf",
    "--output_file", f"{VEP_DIR}/output/{ANC}_R{RELEASE}_{DT.lower()}_vep_output.txt",
    "--refseq",
    "--flag_pick_allele",
    "--no_stats",
    "--fork", "4",
    "--hgvs", # get hgvs annotations 
    "--use_transcript_ref",
    "--assembly", "GRCh38",
    "--fasta", f"{HOME}/.vep/homo_sapiens/115_GRCh38/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz",
    "--force_overwrite"
]

try:
    subprocess.run(vep_cmd, check=True)
except Exception as e:
    print(f"Error: {e}")

2025-12-11 03:25:58 - INFO: BAM-edited cache detected, enabling --use_transcript_ref; use --use_given_ref to override this


In [37]:
# remove header lines
! grep -v "##" {VEP_DIR}/output/{ANC}_R{RELEASE}_{DT.lower()}_vep_output.txt > {WORK_DIR}/{ANC}_R{RELEASE}_{DT.lower()}_vep_output.txt

In [38]:
# filter to variants flagged by pick
! awk 'NR==1 || /PICK/' {WORK_DIR}/{ANC}_R{RELEASE}_{DT.lower()}_vep_output.txt > {WORK_DIR}/{ANC}_R{RELEASE}_{DT.lower()}_vep_output_filtered_pick.txt

In [39]:
# Load VEP output in pd df
# Skip commented lines
# Select columns of interest
gene = pd.read_csv(f"{WORK_DIR}/{ANC}_R{RELEASE}_{DT.lower()}_vep_output_filtered_pick.txt", sep='\t', engine='pyarrow')
gene.head()

,#Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,Existing_variation,Extra
0,chr21:33542509:A:G,21:33542509,G,6651,NM_138927.4,Transcript,upstream_gene_variant,-,-,-,-,-,-,IMPACT=MODIFIER;DISTANCE=529;STRAND=1;PICK=1;G...
1,chr21:33542514:TA:T,21:33542515,-,6651,NM_138927.4,Transcript,upstream_gene_variant,-,-,-,-,-,-,IMPACT=MODIFIER;DISTANCE=523;STRAND=1;PICK=1;G...
2,chr21:33542523:AAAT:A,21:33542524-33542526,-,6651,NM_138927.4,Transcript,upstream_gene_variant,-,-,-,-,-,-,IMPACT=MODIFIER;DISTANCE=512;STRAND=1;PICK=1;G...
3,chr21:33542525:AT:A,21:33542526,-,6651,NM_138927.4,Transcript,upstream_gene_variant,-,-,-,-,-,-,IMPACT=MODIFIER;DISTANCE=512;STRAND=1;PICK=1;G...
4,chr21:33542591:C:G,21:33542591,G,6651,NM_138927.4,Transcript,upstream_gene_variant,-,-,-,-,-,-,IMPACT=MODIFIER;DISTANCE=447;STRAND=1;PICK=1;G...


In [40]:
# ref seq ID for ITSN1 is 6453, filter for it to keep only ITSN1 variants
gene_subset = gene[gene["Gene"] == "6453"].copy()
print("ITSN1 variants: ", gene_subset.shape[0])

ITSN1 variants:  10020


In [41]:
# split extra information into its own dataframe
extra_df = gene_subset["Extra"].str.split(";", expand=True).copy()
extra_df = extra_df.rename(columns={0:"Impact", 1:"Distance", 2:"Strand", 3:"PICK", 4:"Given_Ref", 5:"Used_Ref"})
extra_df

,Impact,Distance,Strand,PICK,Given_Ref,Used_Ref,6
3669,IMPACT=MODIFIER,DISTANCE=713,STRAND=1,PICK=1,GIVEN_REF=T,USED_REF=T,None
3670,IMPACT=MODIFIER,DISTANCE=694,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None
3671,IMPACT=MODIFIER,DISTANCE=685,STRAND=1,PICK=1,GIVEN_REF=G,USED_REF=G,None
3672,IMPACT=MODIFIER,DISTANCE=684,STRAND=1,PICK=1,GIVEN_REF=T,USED_REF=T,None
3673,IMPACT=MODIFIER,DISTANCE=675,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None
...,...,...,...,...,...,...,...
13684,IMPACT=MODIFIER,DISTANCE=3436,STRAND=1,PICK=1,GIVEN_REF=G,USED_REF=G,None
13685,IMPACT=MODIFIER,DISTANCE=3482,STRAND=1,PICK=1,GIVEN_REF=A,USED_REF=A,None
13686,IMPACT=MODIFIER,DISTANCE=3554,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None
13687,IMPACT=MODIFIER,DISTANCE=3587,STRAND=1,PICK=1,GIVEN_REF=T,USED_REF=T,None


In [42]:
# get hgvs annotations
hgvs_df = extra_df[6].str.split(":", expand=True).copy()
hgvs_df = hgvs_df.rename(columns={1:"HGVS"})
hgvs_df

,0,HGVS
3669,None,None
3670,None,None
3671,None,None
3672,None,None
3673,None,None
...,...,...
13684,None,None
13685,None,None
13686,None,None
13687,None,None


In [43]:
# drop unnecessary columns
gene_subset = gene_subset.drop(columns=["Extra", "cDNA_position","CDS_position", "Protein_position", "Amino_acids", "Codons", "Existing_variation"])
gene_subset

,#Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence
3669,chr21:33641788:T:C,21:33641788,C,6453,NM_003024.3,Transcript,upstream_gene_variant
3670,chr21:33641807:C:G,21:33641807,G,6453,NM_003024.3,Transcript,upstream_gene_variant
3671,chr21:33641816:G:A,21:33641816,A,6453,NM_003024.3,Transcript,upstream_gene_variant
3672,chr21:33641817:T:G,21:33641817,G,6453,NM_003024.3,Transcript,upstream_gene_variant
3673,chr21:33641826:C:T,21:33641826,T,6453,NM_003024.3,Transcript,upstream_gene_variant
...,...,...,...,...,...,...,...
13684,chr21:33903297:G:A,21:33903297,A,6453,NM_003024.3,Transcript,downstream_gene_variant
13685,chr21:33903343:A:C,21:33903343,C,6453,NM_003024.3,Transcript,downstream_gene_variant
13686,chr21:33903415:C:G,21:33903415,G,6453,NM_003024.3,Transcript,downstream_gene_variant
13687,chr21:33903448:T:A,21:33903448,A,6453,NM_003024.3,Transcript,downstream_gene_variant


In [ ]:
# save df of extra information for appending later
final_extra_df = pd.concat([gene_subset, extra_df.drop(columns=[6]), hgvs_df.drop(columns=0)], axis=1).rename(columns={"#Uploaded_variation":"SNP"})
final_extra_df.to_csv(f"{WORK_DIR}/vep_annotations_filtered_pick_extra_columns.txt", sep="\t", index=False,)

In [45]:
final_extra_df

,SNP,Location,Allele,Gene,Feature,Feature_type,Consequence,Impact,Distance,Strand,PICK,Given_Ref,Used_Ref,HGVS
3669,chr21:33641788:T:C,21:33641788,C,6453,NM_003024.3,Transcript,upstream_gene_variant,IMPACT=MODIFIER,DISTANCE=713,STRAND=1,PICK=1,GIVEN_REF=T,USED_REF=T,None
3670,chr21:33641807:C:G,21:33641807,G,6453,NM_003024.3,Transcript,upstream_gene_variant,IMPACT=MODIFIER,DISTANCE=694,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None
3671,chr21:33641816:G:A,21:33641816,A,6453,NM_003024.3,Transcript,upstream_gene_variant,IMPACT=MODIFIER,DISTANCE=685,STRAND=1,PICK=1,GIVEN_REF=G,USED_REF=G,None
3672,chr21:33641817:T:G,21:33641817,G,6453,NM_003024.3,Transcript,upstream_gene_variant,IMPACT=MODIFIER,DISTANCE=684,STRAND=1,PICK=1,GIVEN_REF=T,USED_REF=T,None
3673,chr21:33641826:C:T,21:33641826,T,6453,NM_003024.3,Transcript,upstream_gene_variant,IMPACT=MODIFIER,DISTANCE=675,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13684,chr21:33903297:G:A,21:33903297,A,6453,NM_003024.3,Transcript,downstream_gene_variant,IMPACT=MODIFIER,DISTANCE=3436,STRAND=1,PICK=1,GIVEN_REF=G,USED_REF=G,None
13685,chr21:33903343:A:C,21:33903343,C,6453,NM_003024.3,Transcript,downstream_gene_variant,IMPACT=MODIFIER,DISTANCE=3482,STRAND=1,PICK=1,GIVEN_REF=A,USED_REF=A,None
13686,chr21:33903415:C:G,21:33903415,G,6453,NM_003024.3,Transcript,downstream_gene_variant,IMPACT=MODIFIER,DISTANCE=3554,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None
13687,chr21:33903448:T:A,21:33903448,A,6453,NM_003024.3,Transcript,downstream_gene_variant,IMPACT=MODIFIER,DISTANCE=3587,STRAND=1,PICK=1,GIVEN_REF=T,USED_REF=T,None


In [46]:
# edit dataframe to create variants to keep file that matches plink format
gene_subset["Location"] = gene_subset["Location"].str.replace("21:", "")
variation_df = gene_subset["#Uploaded_variation"].str.split(":", expand=True)
location_df = gene_subset["Location"].str.split("-", expand=True)

location_df.columns = ["Start", "End"]
location_df["End"] = location_df.apply(lambda row: row["Start"] if row["End"] is None else row["End"], axis=1)

variation_df.columns = ["Chr", "Start", "Ref", "Alt"]
variation_df.drop(columns=["Start"], inplace=True)

fixed_columns = pd.concat([variation_df, location_df], axis=1)
gene_df = pd.concat([fixed_columns,gene_subset], axis=1)

# drop unnecessary columns
gene_df.drop(columns=["Location", "Feature", "Feature_type"], axis=1, inplace=True)

gene_df["Gene"] = "ITSN1"
gene_df.rename(columns={"#Uploaded_variation":"SNP"}, inplace=True)

In [47]:
# get value counts to determine how many total annotated
variant_counts = gene_df["Consequence"].value_counts().rename_axis('Consequence').reset_index(name='counts').assign(
    Consequence=lambda x: x['Consequence'].str.replace('_', ' '))
variant_counts

,Consequence,counts
0,intron variant,9240
1,3 prime UTR variant,422
2,downstream gene variant,148
3,synonymous variant,60
4,missense variant,60
5,upstream gene variant,51
6,5 prime UTR variant,14
7,"splice polypyrimidine tract variant,intron var...",11
8,"splice region variant,splice polypyrimidine tr...",10
9,"splice region variant,intron variant",2


In [48]:
# write value_counts output to file
variant_counts.to_csv(f"{WORK_DIR}/{ANC}_{GENE}_all_vep_consequences.txt", sep="\t", index=False)

In [49]:
# filter consequences to only missense, splicing, frameshift and stopgain, and other high priority consequences
# Source: https://useast.ensembl.org/info/genome/variation/prediction/predicted_data.html 

vep_coding_variants = gene_df[gene_df["Consequence"].str.contains('^splice_donor_variant|^splice_acceptor_variant|^missense|^stop_gained|frame|^protein_altering|transcript_ablation|transcript_amplification', regex=True)].copy()
vep_coding_variants["Consequence"].value_counts()

Consequence
missense_variant    60
Name: count, dtype: int64

In [50]:
# filter out any synonymous variants
final_vep_coding_variants = vep_coding_variants[~(vep_coding_variants["Consequence"].str.contains('synonymous_variant', regex=False))].copy()
final_vep_coding_variants["Consequence"].value_counts()

Consequence
missense_variant    60
Name: count, dtype: int64

In [51]:
final_vep_coding_variants

,Chr,Ref,Alt,Start,End,SNP,Allele,Gene,Consequence
7254,chr21,G,A,33735084,33735084,chr21:33735084:G:A,A,ITSN1,missense_variant
7255,chr21,A,G,33735115,33735115,chr21:33735115:A:G,G,ITSN1,missense_variant
7257,chr21,C,T,33735157,33735157,chr21:33735157:C:T,T,ITSN1,missense_variant
7811,chr21,G,A,33750157,33750157,chr21:33750157:G:A,A,ITSN1,missense_variant
7812,chr21,C,T,33750172,33750172,chr21:33750172:C:T,T,ITSN1,missense_variant
7813,chr21,C,G,33750224,33750224,chr21:33750224:C:G,G,ITSN1,missense_variant
7814,chr21,G,A,33750235,33750235,chr21:33750235:G:A,A,ITSN1,missense_variant
7816,chr21,C,T,33750263,33750263,chr21:33750263:C:T,T,ITSN1,missense_variant
7818,chr21,A,G,33750294,33750294,chr21:33750294:A:G,G,ITSN1,missense_variant
7819,chr21,A,C,33750297,33750297,chr21:33750297:A:C,C,ITSN1,missense_variant


In [52]:
# Save ids in PLINK format
final_vep_coding_variants["SNP"].to_csv(f"{WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt", sep="\t",  index=False, header=False)

In [53]:
# save consequences annotation file for later
final_vep_coding_variants.to_csv(f"{WORK_DIR}/{ANC}_{GENE}_vep_coding_variant_annotations.txt",  sep="\t",  index=False)

## Burden Analyses using RVTests


In [54]:
PFILE_PREFIX = f"chr{CHROM}_{ANC}"

In [55]:
# get hg38 refFlat file from ucsc
! wget -nc -O /home/jupyter/refFlat.txt.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz
! gunzip -q -f -k /home/jupyter/refFlat.txt.gz

File ‘/home/jupyter/refFlat.txt.gz’ already there; not retrieving.


In [56]:
# make a pheno file for plink input
! cut -f 1,2,5 {WORK_DIR}/{ANC}_covariate_file.txt > {WORK_DIR}/{ANC}_pheno.txt

In [57]:
# Convert the files from Plink 2.0 to Plink 1.9 format 
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/{PFILE_PREFIX} \
--make-bed \
--pheno {WORK_DIR}/{ANC}_pheno.txt \
--pheno-name PHENO \
--max-alleles 2 \
--out {WORK_DIR}/{PFILE_PREFIX}

PLINK v2.0.0-a.6.24LM 64-bit Intel (15 Sep 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR.log.
Options in effect:
  --make-bed
  --max-alleles 2
  --out /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR
  --pfile /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR
  --pheno /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_pheno.txt
  --pheno-name PHENO

Start time: Thu Dec 11 03:28:33 2025
12962 MiB RAM detected, ~9461 available; reserving 6481 MiB for main workspace.
Using up to 2 compute threads.
41021 samples (17013 females, 24008 males; 41021 founders) loaded from
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR.psam.
17377 variants loaded from
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR.pvar.
1 binary phenotype loaded (27691 cases, 13330 controls).
17377 variants re

done.
End time: Thu Dec 11 03:28:33 2025


In [ ]:
# extract variants as a vcf
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{PFILE_PREFIX} \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--extract {WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt \
--recode vcf-iid \
--out {WORK_DIR}/{ANC}_{GENE}.coding_nonsyn

PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.coding_nonsyn.log.
Options in effect:
  --bfile /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR
  --extract /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.all_variants_toKeep.txt
  --keep /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR.samplestoKeep
  --out /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.coding_nonsyn
  --recode vcf-iid

12962 MB RAM detected; reserving 6481 MB for main workspace.
17377 variants loaded from .bim file.
41021 people (24008 males, 17013 females) loaded from .fam.
41021 phenotype values loaded from .fam.
--extract: 60 variants remaining.
--keep: 41021 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 41021 founde

20%21%22%23%24%25%26%27%28%29%30%31%32%33%34%35%36%37%38%39%40%41%42%43%44%45%46%47%48%49%50%51%52%53%54%55%56%57%58%59%60%61%62%63%64%65%66%67%68%69%70%71%72%73%74%75%76%77%78%79%80%81%82%83%84%85%86%87%88%89%90%91%92%93%94%95%96%97%98%99%done.


In [59]:
### Bgzip and Tabix (zip and index the file)
! bgzip -f {WORK_DIR}/{ANC}_{GENE}.coding_nonsyn.vcf
! tabix -f -p vcf {WORK_DIR}/{ANC}_{GENE}.coding_nonsyn.vcf.gz

In [ ]:
# create cov file that matches rvtest pheno file header
rv_df = pd.read_csv(f'{WORK_DIR}/{ANC}_covariate_file.txt', sep="\t")
rv_df.columns = rv_df.columns.str.lower()
rv_df["fid"] = rv_df["#fid"]
rv_df["fatid"] = 0
rv_df["matid"] = 0
rv_df = rv_df[["fid", "iid", "fatid", "matid", "sex", "pheno", "age", "pc1", "pc2", "pc3", "pc4", "pc5"]]
rv_df.to_csv(f'{WORK_DIR}/rvtests_covariate_file.txt', sep='\t', index=False, header=True)

In [62]:
## RVtests with covariates 
! /home/jupyter/tools/rvtests/executable/rvtest --noweb --hide-covar \
--out {WORK_DIR}/{ANC}_{GENE}.burden.coding_nonsyn \
--kernel skat,skato \
--inVcf {WORK_DIR}/{ANC}_{GENE}.coding_nonsyn.vcf.gz \
--pheno {WORK_DIR}/rvtests_covariate_file.txt \
--pheno-name pheno \
--gene {GENE} \
--geneFile /home/jupyter/refFlat.txt \
--covar {WORK_DIR}/rvtests_covariate_file.txt \
--covar-name sex,pc1,pc2,pc3,pc4,pc5

# --out : Name of output 
# --burden cmc --kernel skato: tests to run 
# --inVcf : VCF file 
# --gene: gene name (if only looking at one or a few)
# --geneFile refFlat.txt
# --pheno :  covar file
# --mpheno : # column that has phenotype information
# --pheno-name : column name with phenotype in file
# --covar : covar file
# --freqUpper : optional, MAF cut-off
# --covar-name : covariates, listed by column name, separated by commas (no spaces between commas)
## 1=controls; 2=cases

Thank you for using rvtests (version: 20190205, git: c86e589efef15382603300dc7f4c3394c82d69b8)
  For documentations, refer to http://zhanxw.github.io/rvtests/
  For questions and comments, plase send to Xiaowei Zhan <zhanxw@umich.edu>
  For bugs and feature requests, please submit at: https://github.com/zhanxw/rvtests/issues

The following parameters are available.  Ones with "[]" are in effect:

Available Options
      Basic Input/Output:
                          --inVcf [/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.coding_nonsyn.vcf.gz]
                          --inBgen [], --inBgenSample [], --inKgg []
                          --out [/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.burden.coding_nonsyn]
                          --outputRaw
       Specify Covariate:
                          --covar [/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/rvtests_covariate_file.txt]
                          --covar-name [sex,pc1,pc2,pc3,pc4,pc5

[INFO]	Loaded [ 41021 ] samples from genotype files


[INFO]	Loaded [ 41021 ] sample phenotypes


[INFO]	Begin to read covariate file


[INFO]	Loaded 24008 male, 17013 female and 0 sex-unknown samples from /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/rvtests_covariate_file.txt


[INFO]	Loaded 27691 cases, 13330 controls, and 0 missing phenotypes
[WARN]	-- Enabling binary phenotype mode -- 
[INFO]	Analysis begins with [ 41021 ] samples...


[INFO]	SKAT test significance will be evaluated using 10000 permutations at alpha = 0.05 weight = Beta[beta1 = 1.00, beta2 = 25.00]
[INFO]	SKAT-O test significance will be evaluated using weight = Beta[beta1 = 1.00, beta2 = 25.00]


[INFO]	Loaded [ 1 ] genes.
[INFO]	Impute missing genotype to mean (by default)
[INFO]	Analysis started


## Case/Control Frequencies

### Run PLINK2 GLM

In [66]:
# run logistic regression for pvals and odds ratios
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/{PFILE_PREFIX} \
--adjust \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--pheno {WORK_DIR}/{ANC}_pheno.txt \
--extract {WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt \
--ci 0.95 \
--covar {WORK_DIR}/{ANC}_covariate_file.txt \
--covar-name SEX,PC1,PC2,PC3,PC4,PC5 \
--covar-variance-standardize \
--glm hide-covar omit-ref firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+gcountcc,+err \
--out {WORK_DIR}/{ANC}_{GENE}

PLINK v2.0.0-a.6.24LM 64-bit Intel (15 Sep 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.log.
Options in effect:
  --adjust
  --ci 0.95
  --covar /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_covariate_file.txt
  --covar-name SEX,PC1,PC2,PC3,PC4,PC5
  --covar-variance-standardize
  --extract /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.all_variants_toKeep.txt
  --glm hide-covar omit-ref firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+gcountcc,+err
  --keep /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR.samplestoKeep
  --out /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1
  --pfile /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR
  --pheno /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_pheno.txt

Start time: Thu Dec 11 03

12962 MiB RAM detected, ~9522 available; reserving 6481 MiB for main workspace.
Using up to 2 compute threads.


41021 samples (17013 females, 24008 males; 41021 founders) loaded from
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR.psam.


17377 variants loaded from
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR.pvar.
2 binary phenotypes loaded.
--extract: 60 variants remaining.


--keep: 41021 samples remaining.


6 covariates loaded from /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_covariate_file.txt.
41021 samples (17013 females, 24008 males; 41021 founders) remaining after main
filters.
--covar-variance-standardize: 6 covariates transformed.
60 variants remaining after main filters.


--glm logistic-Firth hybrid regression on phenotype 'PHENO1': 0%

0%

done.
Results written to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.PHENO1.glm.logistic.hybrid .
--adjust: Genomic inflation est. lambda (based on median chisq) = 0.500904.
(Treating lambda as 1 in GC-corrected p-value calculation.)
--adjust values (59 tests) written to
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.PHENO1.glm.logistic.hybrid.adjusted
.
--glm logistic-Firth hybrid regression on phenotype 'PHENO': 0%0%

done.
Results written to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.PHENO.glm.logistic.hybrid .
--adjust: Genomic inflation est. lambda (based on median chisq) = 0.488784.
(Treating lambda as 1 in GC-corrected p-value calculation.)
--adjust values (60 tests) written to
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.PHENO.glm.logistic.hybrid.adjusted
.
End time: Thu Dec 11 03:29:09 2025


### Run Basic Association Test (fisher and 1df chi-sq)

In [67]:
BFILE_PREFIX = f"chr{CHROM}_{ANC}"

In [68]:
## Run association test using Fisher's exact test
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--pheno {WORK_DIR}/{ANC}_pheno.txt \
--mpheno 1 \
--assoc fisher \
--ci 0.95 \
--allow-no-sex \
--out {WORK_DIR}/{ANC}_{GENE}

PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.log.
Options in effect:
  --allow-no-sex
  --assoc fisher
  --bfile /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR
  --ci 0.95
  --extract /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.all_variants_toKeep.txt
  --keep /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR.samplestoKeep
  --mpheno 1
  --out /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1
  --pheno /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_pheno.txt

12962 MB RAM detected; reserving 6481 MB for main workspace.
17377 variants loaded from .bim file.


41021 people (24008 males, 17013 females) loaded from .fam.


41021 phenotype values present after --pheno.
--extract: 60 variants remaining.


--keep: 41021 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 41021 founders and 0 nonfounders present.
Calculating allele frequencies... 0%1%2%3%4%5%6%7%8%9%10%11%12%13%14%15%16%17%18%19%20%21%22%23%24%

25%26%27%28%29%30%31%32%33%34%35%36%37%38%39%40%41%42%43%44%45%46%47%48%49%

50%51%52%53%54%55%56%57%58%59%

60%61%62%63%64%65%66%67%68%69%70%71%72%73%74%75%76%77%78%79%80%81%82%83%84%85%86%87%88%89%90%91%92%93%94%95%96%97%98%99% done.
Total genotyping rate is 0.999876.
60 variants and 41021 people pass filters and QC.
Among remaining phenotypes, 27691 are cases and 13330 are controls.
Writing C/C --assoc report to
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.assoc.fisher
... 0%done.


### Get variant frequencies

In [69]:
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--allow-no-sex \
--freq \
--out {WORK_DIR}/{ANC}_{GENE}

PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.log.
Options in effect:
  --allow-no-sex
  --bfile /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR
  --extract /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.all_variants_toKeep.txt
  --freq
  --keep /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR.samplestoKeep
  --out /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1

12962 MB RAM detected; reserving 6481 MB for main workspace.
17377 variants loaded from .bim file.
41021 people (24008 males, 17013 females) loaded from .fam.
41021 phenotype values loaded from .fam.
--extract: 60 variants remaining.
--keep: 41021 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 41021 founders and 0 nonfounders 

### Merge files together

In [70]:
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--pheno {WORK_DIR}/{ANC}_pheno.txt \
--pheno-name PHENO \
--allow-no-sex \
--recode A \
--out {WORK_DIR}/{ANC}_{GENE}

PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.log.
Options in effect:
  --allow-no-sex
  --bfile /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/chr21_EUR
  --extract /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1.all_variants_toKeep.txt
  --keep /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR.samplestoKeep
  --out /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_ITSN1
  --pheno /home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/EUR/EUR_pheno.txt
  --pheno-name PHENO
  --recode A

12962 MB RAM detected; reserving 6481 MB for main workspace.
17377 variants loaded from .bim file.
41021 people (24008 males, 17013 females) loaded from .fam.
41021 phenotype values present after --pheno.
--extract: 60 variants remaining.


--keep: 41021 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 41021 founders and 0 nonfounders present.
Calculating allele frequencies... 0%1%2%3%4%5%6%7%8%9%10%11%12%13%14%15%16%17%18%19%20%21%22%23%24%25%26%27%28%29%30%31%32%33%34%35%36%37%38%39%40%41%42%43%44%45%46%47%48%49%50%51%52%53%54%55%56%57%58%59%60%61%62%63%64%65%66%67%68%69%70%71%72%73%74%75%76%77%78%79%80%81%82%83%84%85%86%87%88%89%90%91%92%93%94%95%96%97%98%99% done.
Total genotyping rate is 0.999876.
60 variants and 41021 people pass filters and QC.
Among remaining phenotypes, 27691 are cases and 13330 are controls.
--recode A to
/home/jupyter/ITSN1_R11_Results_12112025/ITSN1_IMP/E

In [71]:
### merge output files

ASSOC_FILE = f"{WORK_DIR}/{ANC}_{GENE}.assoc.fisher"
RECODE_FILE = f"{WORK_DIR}/{ANC}_{GENE}.raw"
GLM_HYBRID_FILE = f"{WORK_DIR}/{ANC}_{GENE}.PHENO.glm.logistic.hybrid"
GLM_ADJUST_FILE = f"{WORK_DIR}/{ANC}_{GENE}.PHENO.glm.logistic.hybrid.adjusted"
FREQ_FILE = f"{WORK_DIR}/{ANC}_{GENE}.frq"


log_hybrid = pd.read_csv(GLM_HYBRID_FILE, sep="\s+", usecols=["ID", "A1", "A1_FREQ", "A1_CT", "ALLELE_CT", "P", "OR", "LOG(OR)_SE", "L95", "U95", "FIRTH?", "ERRCODE"])
log_hybrid.rename(columns={"ID":"SNP"}, inplace=True)

assoc_adjusted = pd.read_csv(GLM_ADJUST_FILE,  sep="\s+", usecols=["ID", "BONF"])
assoc_adjusted.rename(columns={"ID":"SNP"}, inplace=True)

assoc = pd.read_csv(ASSOC_FILE, sep="\s+", usecols=["SNP", "P", "F_A", "F_U"])
assoc.rename(columns={"P": "P_fisher"}, inplace=True)


freq = pd.read_csv(FREQ_FILE, sep='\s+', usecols=['SNP', 'MAF'])
df = pd.merge(log_hybrid, assoc_adjusted, on="SNP", how="right")
df = pd.merge(df, freq, on="SNP", how="left")
#merge freq with df
freq_assoc = pd.merge(assoc, df, on="SNP", how="left")

#read in recode file
recode = pd.read_csv(RECODE_FILE, sep="\s+")

# Pre-filter the dataset
cases_data = recode[recode["PHENOTYPE"] == 2]
controls_data = recode[recode["PHENOTYPE"] == 1]
# Make a list from the column names
column_names = recode.columns.tolist()

# Drop the first 6 columns to keep the variants 
variants = column_names[6:]
results = []
for variant in variants:
    # For cases
    hom_cases = cases_data[cases_data[variant] == 2].shape[0]
    het_cases = cases_data[cases_data[variant] == 1].shape[0]
    total_cases = cases_data.shape[0]
    # For controls
    hom_controls = controls_data[controls_data[variant] == 2].shape[0]
    het_controls = controls_data[controls_data[variant] == 1].shape[0]
    total_controls = controls_data.shape[0]
    results.append({
        "Variant": variant,
        "Hom Cases": hom_cases,
        "Het Cases": het_cases,
        "Total Cases": total_cases,
        "Hom Controls": hom_controls,
        "Het Controls": het_controls,
        "Total Controls": total_controls,
    })

# Return results
df_results = pd.DataFrame(results)
df_results["SNP"] = df_results["Variant"].apply(lambda x: x.rsplit("_", 1)[0])
df_results = df_results.drop("Variant", axis=1)

# Merge with assoc results 
full_results = pd.merge(freq_assoc, df_results, on="SNP", how="left")


# append variant annotation
annotations = pd.read_csv(f"{WORK_DIR}/vep_annotations_filtered_pick_extra_columns.txt", sep="\t", usecols=["Consequence", "SNP", "HGVS"])
full_results_annotated = pd.merge(full_results, annotations, on="SNP", how="left")
full_results_annotated.head()
#subset to only columns to keep

clean_full_results = full_results_annotated[["SNP", "Consequence", "HGVS", "A1", "A1_CT", "A1_FREQ", "FIRTH?",
                                             "P", "P_fisher", "BONF", "OR", "LOG(OR)_SE","L95", "U95",
                                  "F_A", "F_U", "MAF",
                                   "Hom Cases", "Het Cases", "Total Cases", 
                                   "Hom Controls","Het Controls", "Total Controls", "ERRCODE"
                                   ]].copy()



clean_full_results.insert(1, "Ancestry", ANC)

clean_full_results.head()

,SNP,Ancestry,Consequence,HGVS,A1,A1_CT,A1_FREQ,FIRTH?,P,P_fisher,...,F_A,F_U,MAF,Hom Cases,Het Cases,Total Cases,Hom Controls,Het Controls,Total Controls,ERRCODE
0,chr21:33735084:G:A,EUR,missense_variant,p.Val76Met,A,1,0.000012,Y,0.962659,1.0000,...,0.000018,0.000000,0.000012,0,1,27691,0,0,13330,.
1,chr21:33735115:A:G,EUR,missense_variant,p.Lys86Arg,G,1,0.000012,Y,0.362247,0.3249,...,0.000000,0.000038,0.000012,0,0,27691,0,1,13330,.
2,chr21:33735157:C:T,EUR,missense_variant,p.Pro100Leu,T,2,0.000024,N,0.453683,0.5443,...,0.000018,0.000038,0.000024,0,1,27691,0,1,13330,.
3,chr21:33750157:G:A,EUR,missense_variant,p.Ala121Thr,A,32,0.000391,N,0.847069,1.0000,...,0.000398,0.000376,0.000391,0,22,27691,0,10,13330,.
4,chr21:33750172:C:T,EUR,missense_variant,p.Leu126Phe,T,1,0.000012,Y,0.328080,0.3250,...,0.000000,0.000038,0.000012,0,0,27691,0,1,13330,.


In [72]:
# Look at significant SNPs, if any 
sig_freq = clean_full_results[clean_full_results['P']<0.05]
sig_snps = sig_freq['SNP'].tolist()
sig_df_results = clean_full_results[clean_full_results['SNP'].isin(sig_snps)]
sig_df_results

,SNP,Ancestry,Consequence,HGVS,A1,A1_CT,A1_FREQ,FIRTH?,P,P_fisher,...,F_A,F_U,MAF,Hom Cases,Het Cases,Total Cases,Hom Controls,Het Controls,Total Controls,ERRCODE
38,chr21:33814053:C:T,EUR,missense_variant,p.Pro903Leu,T,65,0.000793,N,0.043451,0.08406,...,0.000668,0.001051,0.000793,0,37,27691,0,28,13330,.


In [73]:
# save files to working directory
clean_full_results.to_csv(f'{WORK_DIR}/{ANC}_{GENE}_GP2_R{RELEASE}.fullVariantInformation.txt', sep="\t", index=False)
sig_df_results.to_csv(f'{WORK_DIR}/{ANC}_{GENE}_GP2_R{RELEASE}.SignificantVariantInformation.txt' , sep="\t", index=False)